# Predictions

Here we finally deploy our trained models to make predictions on real images.
The program uses Sliding Window method to extract sub-images from the full satellite image.
It then extracts features of those sub-images to determine the presence of waste in them.

In [3]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import time
import Utilities

In [8]:
src_folder_path = '/Users/sinner/Desktop/Batch5'
dest_folder_path = '/Users/sinner/Desktop/Batch_processing_outputs'

In [9]:
# Defining a custom standard scaler to scale features

mean_std_df = pd.read_csv('mean_std_df.csv')

def my_standard_scaler(features):
    scaled_features = []
    for i, value in enumerate(features):
        mean = mean_std_df.iat[i, 1]
        std = mean_std_df.iat[i, 2]
        scaled_features.append((value - mean) / std)
    return np.array(scaled_features)

In [10]:
# Load model
loaded_model = pickle.load(open('trained_models/RandomForestClassifier.sav', 'rb'))
print('Model Loaded')

Model Loaded


In [11]:
# Driver code

start_time = time.time()

# iterating in source folder
for filename in os.listdir(src_folder_path):

    # Check whether filetype is correct
    if not filename.endswith(('.jpg', '.png', '.jpeg')):
        continue

    print(f"Scanning {filename}")

    # load image
    img_bgr = cv2.imread(os.path.join(src_folder_path, filename))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    positiveWindows = []
    for window_coord in Utilities.get_window_coords(img_rgb, 150, 150):

        # get cropped window from img
        # cropped_img = img[top:bottom, left:right]
        window = img_rgb[round(window_coord[0]):round(window_coord[1]), round(window_coord[2]):round(window_coord[3])]

        # get features of window
        features = Utilities.get_features(window)
        # Feature Scaling
        features = my_standard_scaler(features)

        # Predict class using model
        predicted_class = loaded_model.predict(np.reshape(features, (1,-1)))
        # print(predicted_class)

        if predicted_class[0] == 1:
            # append coordinates into positiveWindows list if waste detected
            positiveWindows.append(window_coord)

        overlay_time_start = time.time()
        for window_coord in positiveWindows:
            # Overlay the bounding box on the image
            cv2.rectangle(img_bgr, (window_coord[2], window_coord[0]), (window_coord[3], window_coord[1]), (0, 255, 0), 2)

    cv2.imwrite(os.path.join(dest_folder_path, filename), img_bgr)

print(f"Total time taken: {time.time() - start_time}")

Scanning 0515.jpg
Scanning 0501.jpg
Scanning 0529.jpg
Scanning 0663.jpg
Scanning 0677.jpg
Scanning 0688.jpg
Scanning 0689.jpg
Scanning 0676.jpg
Scanning 0662.jpg
Scanning 0528.jpg
Scanning 0514.jpg
Scanning 0700.jpg
Scanning 0502.jpg
Scanning 0516.jpg
Scanning 0674.jpg
Scanning 0660.jpg
Scanning 0648.jpg
Scanning 0649.jpg
Scanning 0661.jpg
Scanning 0675.jpg
Scanning 0517.jpg
Scanning 0503.jpg
Scanning 0507.jpg
Scanning 0513.jpg
Scanning 0659.jpg
Scanning 0671.jpg
Scanning 0665.jpg
Scanning 0664.jpg
Scanning 0670.jpg
Scanning 0658.jpg
Scanning 0512.jpg
Scanning 0506.jpg
Scanning 0538.jpg
Scanning 0510.jpg
Scanning 0504.jpg
Scanning 0666.jpg
Scanning 0672.jpg
Scanning 0699.jpg
Scanning 0698.jpg
Scanning 0673.jpg
Scanning 0667.jpg
Scanning 0505.jpg
Scanning 0511.jpg
Scanning 0539.jpg
Scanning 0576.jpg
Scanning 0562.jpg
Scanning 0589.jpg
Scanning 0600.jpg
Scanning 0614.jpg
Scanning 0628.jpg
Scanning 0629.jpg
Scanning 0615.jpg
Scanning 0601.jpg
Scanning 0588.jpg
Scanning 0563.jpg
Scanning 0

[ WARN:0@6353.422] global loadsave.cpp:244 findDecoder imread_('/Users/sinner/Desktop/Batch5/0566.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.7.0) /Users/xperience/GHA-OCV-Python/_work/opencv-python/opencv-python/opencv/modules/imgproc/src/color.cpp:182: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'
